In [2]:
import numpy as np
import time

def evaluate_quantum_model(hbar, a_start):
    """
    1D 量子黎曼共振退火模型 (终点精确锚定 + phi^4 紫外正规化版)
    """
    N = 200         
    L = 3.0         
    T_steps = 300   
    a_end = 1.02    
    c_offset = 10.0 
    
    q = np.linspace(-L, L, N, endpoint=False)
    dq = q[1] - q[0]
    dp = 2 * np.pi * hbar / (N * dq)
    p = np.fft.fftfreq(N) * N * dp
    
    T_kin = 0.5 * p**2
    F = np.fft.fft(np.eye(N), axis=0) / np.sqrt(N)
    F_inv = np.fft.ifft(np.eye(N), axis=0) * np.sqrt(N)
    
    exp_iT = np.diag(np.exp(-1j * T_kin / hbar))
    
    t_start = 1.0 / (np.log(1 + c_offset)**2)
    t_end   = 1.0 / (np.log(T_steps + c_offset)**2)
    delta_a = a_start - a_end
    k_opt = delta_a / (t_start - t_end)
    a_dyna = a_end - k_opt * t_end  
    
    U_tot = np.eye(N, dtype=np.complex128)
    
    for t in range(1, T_steps + 1):
        a_t = a_dyna + k_opt / (np.log(t + c_offset)**2)
        
        V_t = -q + q**2 + (a_t / 3.0) * q**3 + 0.05 * q**4
        exp_iV = np.diag(np.exp(-1j * V_t / hbar))
        U_tot = (F_inv @ exp_iT @ F @ exp_iV) @ U_tot 
        
    V_base = -q + q**2 + (a_end / 3.0) * q**3 + 0.05 * q**4
    H_base = F_inv @ np.diag(T_kin) @ F + np.diag(V_base)
    
    evals, evecs = np.linalg.eig(U_tot)
    phases = np.angle(evals)
    
    expected_energies = np.zeros(N)
    for i in range(N):
        psi = evecs[:, i]
        expected_energies[i] = np.real(np.vdot(psi, H_base @ psi))

    E_quantum = np.zeros(N)
    for i in range(N):
        m = np.round((expected_energies[i] * T_steps + hbar * phases[i]) / (2 * np.pi * hbar))
        E_quantum[i] = (-hbar * phases[i] + 2 * np.pi * hbar * m) / T_steps

    E_quantum = np.sort(E_quantum)
    E_quantum = E_quantum - E_quantum[0]

    # 【终极精度】：直接拉满到小数点后 16 位的解析真值
    true_zeros = np.array([
        14.1347251417346938, 21.0220396387715550, 25.0108575801456888, 
        30.4248761258595132, 32.9350615877391897, 37.5861781588256713, 
        40.9187190121474951, 43.3270732809149995, 48.0051508811671597, 
        49.7738324776723022
    ])

    scale_k = true_zeros[0] / E_quantum[1] if E_quantum[1] != 0 else 1.0
    E_pred = E_quantum[1:11] * scale_k
    
    err_sum = np.sum(np.abs(E_pred[:6] - true_zeros[:6]))
    return err_sum, E_pred

def run_quantum_radar():
    print("======================================================")
    print(" 🚀 启动半经典精密雷达：寻找 2D 黎曼共振盆地 ")
    print("    [演化终点严格固定: a_end = 1.0200000000000000] ")
    print("    [全精度模式启动: 提取底层 64 位浮点数极限] ")
    print("======================================================\n")
    
    hbar_vals = np.linspace(0.03, 0.10, 80)    
    a_start_vals = np.linspace(1.025, 1.30, 80) 
    
    best_err = float('inf')
    best_params = None
    best_pred = None
    
    total_iters = len(hbar_vals) * len(a_start_vals)
    count = 0
    start_time = time.time()
    
    for hbar in hbar_vals:
        for a_start in a_start_vals:
            count += 1
            try:
                err, pred = evaluate_quantum_model(hbar, a_start)
                if err < best_err:
                    best_err = err
                    best_params = (hbar, a_start)
                    best_pred = pred
                    # 【终极精度输出】
                    print(f"[*] 🏆 突破记录! 进度 {count}/{total_iters} | hbar={hbar:.16f}, a_start={a_start:.16f} | ErrSum(1-6)={err:.16f}")
            except Exception as e:
                pass
                
    print("\n==================== 🏁 高精度扫描完成 ====================")
    print(f"耗时: {time.time() - start_time:.2f} 秒")
    print(f"拓扑不动点 (Fixed Point): a_end = 1.0200000000000000")
    print(f"最佳物理参数: hbar = {best_params[0]:.16f}, a_start = {best_params[1]:.16f}")
    print(f"前 6 个零点绝对误差和 (Lock-in Error): {best_err:.16f}")
    
    true_zeros = [
        14.1347251417346938, 21.0220396387715550, 25.0108575801456888, 
        30.4248761258595132, 32.9350615877391897, 37.5861781588256713, 
        40.9187190121474951, 43.3270732809149995, 48.0051508811671597, 
        49.7738324776723022
    ]
    print("\n[最佳参数预测谱线对比 (Top 1)]")
    for i in range(10):
        err = abs(best_pred[i] - true_zeros[i])
        print(f"N={i+1:2d} | 真值: {true_zeros[i]:.16f} | 预测: {best_pred[i]:.16f} | 误差: {err:.16f}")

if __name__ == '__main__':
    run_quantum_radar()

 🚀 启动半经典精密雷达：寻找 2D 黎曼共振盆地 
    [演化终点严格固定: a_end = 1.0200000000000000] 
    [全精度模式启动: 提取底层 64 位浮点数极限] 

[*] 🏆 突破记录! 进度 1/6400 | hbar=0.0300000000000000, a_start=1.0249999999999999 | ErrSum(1-6)=8.5106132694210270
[*] 🏆 突破记录! 进度 40/6400 | hbar=0.0300000000000000, a_start=1.1607594936708860 | ErrSum(1-6)=3.0972116463441175

==================== 🏁 高精度扫描完成 ====================
耗时: 2409.05 秒
拓扑不动点 (Fixed Point): a_end = 1.0200000000000000
最佳物理参数: hbar = 0.0300000000000000, a_start = 1.1607594936708860
前 6 个零点绝对误差和 (Lock-in Error): 3.0972116463441175

[最佳参数预测谱线对比 (Top 1)]
N= 1 | 真值: 14.1347251417346946 | 预测: 14.1347251417346946 | 误差: 0.0000000000000000
N= 2 | 真值: 21.0220396387715560 | 预测: 20.7353621966014465 | 误差: 0.2866774421701095
N= 3 | 真值: 25.0108575801456894 | 预测: 25.1193752945294868 | 误差: 0.1085177143837974
N= 4 | 真值: 30.4248761258595124 | 预测: 30.8893070147224833 | 误差: 0.4644308888629709
N= 5 | 真值: 32.9350615877391917 | 预测: 33.2905870904173469 | 误差: 0.3555255026781552
N= 6 | 真值: 37.5861